# Step 10: the KV cache

This notebook goes with step 10 of `vak`. It writes with a small GPT, with and without a KV cache, and counts what
the cache saves and what it costs. `uv run vak cache` does the same with the trained GPT.

Before this: step 9 trains the GPT, and it writes by reading every token again at every step. This is the last
step of `vak`.

| Section | Idea | Used in |
|---|---|---|
| 1 | Prefill: read the prompt once, and keep each block's keys and values | `vak cache` |
| 2 | Decode: one new token reads the saved keys, with the same result | `vak cache` |
| 3 | The memory bill: what the cache holds per token | `vak cache` |

## 1. Prefill

At each step of writing, a GPT without a cache reads every token again. But the keys and values of the earlier
tokens never change: the causal mask means a token never reads a later one. A **KV cache** keeps them. The first
step, the **prefill**, reads the whole prompt once and saves each block's keys and values.

In [1]:
import torch
from medha.model import GPT, Config, KVCache

config = Config(vocab_size=7, context=8, width=8, heads=2, layers=2)
torch.manual_seed(0)
model = GPT(config)
prompt = [0, 1, 2, 3]  # <|endoftext|> the cat sat
cache = KVCache(config.layers)
logits = model.next_logits(prompt, cache)
print("tokens in the cache:", cache.length)
print("keys of block 0:", tuple(cache.keys[0].shape), "  values of block 0:", tuple(cache.values[0].shape))

tokens in the cache: 4
keys of block 0: (1, 2, 4, 4)   values of block 0: (1, 2, 4, 4)


**What you see**

After the prefill, the cache holds `4` tokens. Each block keeps keys and values of the shape `(1, 2, 4, 4)`:
1 text, 2 heads, 4 tokens, 4 numbers per head. Two blocks, so four such tensors in all.

Used in: `uv run vak cache` starts each text with a prefill of the prompt.

## 2. Decode

Each later step, the **decode**, reads only the newest token. Its query meets the saved keys plus its own, and its
key and value join the cache. The logits must equal those of reading every token again.

In [2]:
ids = prompt + [4]  # the model picked "on"
held_before = cache.length
with_cache = model.next_logits(ids, cache)
without = model.next_logits(ids)  # reads all of ids again
print("tokens in the cache:", cache.length)
print("tokens read: with the cache", cache.length - held_before, " without", len(ids))
print("largest difference in the logits:", f"{float((with_cache - without).abs().max()):.7f}")

tokens in the cache: 5
tokens read: with the cache 1  without 5
largest difference in the logits: 0.0000000


**What you see**

The cache now holds `5` tokens. The logits match the full read: the largest difference is `0.0000000` to seven
decimals. The cached step read 1 token instead of 5. Over a long text the saving grows: without a cache, step n
reads n tokens.

Used in: `uv run vak cache` writes the same text twice, without and with the cache, and checks that the tokens match.

## 3. The memory bill

The cache trades memory for work. Per token, it keeps one key and one value in every block: 2 × blocks × width
numbers, at 4 bytes each in float32.

In [3]:
print("bytes in the cache:", cache.size_in_bytes())
print("by hand: 2 x", config.layers, "blocks x", config.width, "numbers x 4 bytes x", cache.length, "tokens =",
      2 * config.layers * config.width * 4 * cache.length)
real = Config(vocab_size=2_000, context=128, width=128, heads=4, layers=4)
print("the trained GPT, per token:", 2 * real.layers * real.width * 4, "bytes; a full window:",
      2 * real.layers * real.width * 4 * real.context, "bytes")

bytes in the cache: 640
by hand: 2 x 2 blocks x 8 numbers x 4 bytes x 5 tokens = 640
the trained GPT, per token: 4096 bytes; a full window: 524288 bytes


**What you see**

The small cache holds `640` bytes, the same as the hand count. The trained GPT keeps `4096` bytes per token, and a
full window of 128 tokens takes `524288` bytes, half a megabyte. A big model keeps far more per token, which is why
the cache, not the weights, often limits how many texts a server can write at once.

Used in: `uv run vak cache` prints the cache's size at the end of the text.

## Recap

| Idea | Remember | Used next |
|---|---|---|
| Prefill | read the prompt once; keep every block's keys and values | the first step of every cached text |
| Decode | the newest token's query meets the saved keys; the same logits as reading everything again | every later step |
| A full window | positions shift, so every saved key is out of date: the cache starts again | a reason newer models put position inside attention |
| The memory bill | 2 × blocks × width × 4 bytes per token: 4,096 bytes for the trained GPT | the cost a server pays per text |